# TC-WPN — Phase 7: Controlled Experiment 1 (`max_chunks` 1 → 4)

**Component:** R26-DS-012 / TC-WPN — Dulhara Kaushalya (IT22130648)

Phase 6 diagnosed. Phase 6B validated and corrected the diagnosis. This is the **intervention**,
and it is the first time since the meeting that anything is retrained.

The supervisor's instruction, and the part that has not yet been done:

> "මිස් ක්ලැසිෆයි වෙන සාම්පල්ස් මොනවද කියන එක මුලින් බලන්න." — first find which samples are
> misclassified. **Done** (Phase 6).
>
> "එක දෙයක් තමයි ඔයාට බලන්න පුළුවන් මේක ඩේටා පැත්තෙන් විසඳන්න පුළුවන්ද" — then see whether it can
> be solved from the data side. **Done** (Phase 6/6B: no strong error stratum, 1.16× against a
> 1.5× floor).
>
> "Then you have to think that the machine learning model has a problem … BERT එකේ embeddings
> කොහොමද ගත්තේ? embedding ටික හරියට තියෙනවද?" — **Done** (Phase 6B staircase).
>
> "ඒක හදාගන්න බලන්න." — **then fix it.** ← this notebook.

The honest status to give him: **the investigation is complete, the fix is not.** Phase 7 runs
one controlled change across five seeds and reports the result either way.

---

## Two corrections that come first (Part 0)

Reviewing the Phase 6B log against the repository source turned up two things that change what
this experiment must do, and what Experiment 2 should be.

### Correction 1 — the frozen episode plans survive a `max_chunks` change

The Phase 6B pre-registration said, and the review repeated:

> "new pkl → new store fingerprint → new episode plans … you cannot simply compare the new model
> against the old frozen plans."

**That is wrong, and it is my error in the Phase 6B text.** `sampler.store_fingerprint` is:

```python
h.update(str(len(store.records)).encode())
for r in store.records[:1000]:
    h.update(str(r["note_id"]).encode())
```

It hashes the **record count and note_ids** — nothing about tokens or chunks. And
`chunk_tokenize` returns `ids[:max_chunks]`, which changes how many chunks a record carries, not
how many records exist or what they are called. So the fingerprint is unchanged, the frozen plans
stay valid, and **the existing five-seed baseline needs no re-scoring**. Part 1 asserts this
rather than assuming it, with a documented fallback if the assertion fails.

This matters practically: it removes an entire baseline re-training run from the experiment.

### Correction 2 — the S2 → S3 gap is not evidence that the prototype stage is broken

The staircase was read as:

> "the largest degradation occurs when going from the learned representation into the full
> episodic TC-WPN inference … I would investigate the S2 → S3 gap."

But S2 and S3 are not comparable as measured, for two reasons visible in the Phase 6B log:

| | S2 probe | S3 TC-WPN |
|---|---|---|
| supervision at inference | logistic regression **fit on 10 290 train patients** | prototypes from **5 support patients per class** |
| notes per test patient | all 3 450 test notes → **1.51/patient** | only queried notes, 2 907 → **1.28/patient** |

A fully-supervised probe trained on 10 290 patients *should* beat a 5-shot prototype. That is the
definition of few-shot learning, not a defect in prototype formation. And the probe additionally
sees ~18% more notes per patient.

The information-matched comparison already exists in the Phase 6 log: the **uniform-weight
counterfactual, 0.7369**, is a 5-shot centroid built on the same S2 embeddings over the same
episodes. So:

```text
S2 embeddings, 10,290-patient supervised probe   0.7611
S2 embeddings, 5-shot centroid, same episodes    0.7369   <- matched to S3
S3 full TC-WPN (5-shot + learned weighting)      0.7379
```

Matched, the prototype stage is **+0.0010 above** a plain 5-shot centroid on its own embeddings —
not 0.0232 below anything. The 0.0242 gap is the **price of the few-shot protocol**, which is a
property of the research design, not a bug to fix.

**Consequence for Experiment 2:** "investigate the S2 → S3 gap" is not the right next move. If
few-shot supervision is what costs 0.024, the questions worth asking are about K, support-set
composition, or whether the episodic protocol is the right fit for this task at all — and that is
a framing question for the paper, not a hyper-parameter to tune.

---

## What Phase 7 changes

```text
max_chunks: 1  ->  4
```

and nothing else. Frozen: cohort, labels, patient splits, episode plans, seeds 42–46, K=5,
architecture, encoder, pooling, projection, temporal weighting, consistency weighting, optimiser,
learning rate, dropout, threshold-selection procedure.

## Why this one, and what it can be worth

From the Phase 6B log: **804 / 2 907 queried notes (27.7%)** carry anxiety terminology only beyond
the first 512-token window; those notes touch **730 / 2 278 patients (32.0%)**, of which **188 are
currently misclassified (24.5% of all errors)**.

That 188 is an **upper bound**, and the direction of the evidence is against the simple story:
incorrect notes are *less* likely to hide anxiety terms past the window (19.96% vs 31.68%,
Cliff's δ = −0.1172). Nothing here predicts success. The experiment is worth running because the
bound is large enough to be measurable and the change is genuinely single-variable.

## Pre-registered decision rule (fixed, from `phase6b_next_experiment.md`)

```text
Seeds     : 42, 43, 44, 45, 46
Selection : validation only; test scored ONCE
Success   : mean ΔAUROC >= +0.020  AND  paired p < 0.05  AND  >= 4/5 seeds improve
Failure   : anything else. Δ below the 0.0199 baseline seed spread is noise.
Either way, the result is reported.
```

Comparator: `tcwpn_full` 0.7377 ± 0.0031 (seeds 42–46: 0.7379, 0.7394, 0.7386, 0.7403, 0.7323).

**0.738 → 0.744 is not success.** One good seed is not success.

## Inputs

| input | why |
|---|---|
| Stage A output | `pkl/`, `plans/`, `cohort_psych_mimic4idx.csv` |
| Stage C / Phase 3B output | frozen `tcwpn_full_k5_seed*` runs for the paired comparison |

Accelerator **GPU T4 x2** (not P100 — sm_60 is not in Kaggle's PyTorch build). Budget: five seeds
× ~47 min ≈ 4 h, plus a longer forward pass because notes now carry up to 4 chunks. Run one seed
per session if the quota is tight.

In [ ]:
# ---------------------------------------------------------------------------
# 0.0  Repo, dependencies, device guard.
# ---------------------------------------------------------------------------
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!git log --oneline -1
!pip install -q -r requirements.txt 2>&1 | tail -2

import os, sys, json, glob, shutil, re
from pathlib import Path
import numpy as np, pandas as pd, torch
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.insert(0, "/kaggle/working/tcwpn_test/src")

ALLOW_CPU_FALLBACK = False
def resolve_device():
    if not torch.cuda.is_available():
        return "cpu"
    cap = torch.cuda.get_device_capability(0); sm = f"sm_{cap[0]}{cap[1]}"
    arches = list(torch.cuda.get_arch_list())
    print(f"GPU   : {torch.cuda.get_device_name(0)} ({sm})")
    print(f"build : {', '.join(arches) or 'unknown'}")
    if arches and sm not in arches:
        print(f"  {sm} NOT in this PyTorch build."); return "unsupported_gpu"
    try:
        (torch.zeros(8, 8, device="cuda") @ torch.zeros(8, 8, device="cuda")).sum().item()
        torch.cuda.synchronize()
    except Exception as e:
        print(f"  kernel test FAILED: {type(e).__name__}: {e}"); return "unsupported_gpu"
    return "cuda"

_d = resolve_device()
if _d == "unsupported_gpu":
    print("\nSWITCH ACCELERATOR TO 'GPU T4 x2' AND RE-RUN.")
    if not ALLOW_CPU_FALLBACK:
        raise SystemExit("unsupported GPU architecture")
    DEVICE = "cpu"
else:
    DEVICE = _d
print("torch", torch.__version__, "| DEVICE", DEVICE)

In [ ]:
# ---------------------------------------------------------------------------
# 0.1  Locate inputs.
# ---------------------------------------------------------------------------
STEM, K = "psych_mimic4idx", 5
SEEDS = [42, 43, 44, 45, 46]
OUT = Path("/kaggle/working/phase7"); OUT.mkdir(parents=True, exist_ok=True)

stage_a = next((p.parent for p in Path("/kaggle/input").rglob("plans")
                if (p.parent / "pkl").exists()), None)
if stage_a is None:
    raise SystemExit("Stage A dataset not found (needs pkl/ and plans/ side by side)")
PKL_OLD, PLAN_DIR = stage_a / "pkl", stage_a / "plans"

hits = sorted(Path("/kaggle/input").rglob(f"cohort_{STEM}.csv"))
if not hits:
    raise SystemExit(f"cohort_{STEM}.csv not found")
COHORT_CSV = hits[0]

RESULTS = Path("/kaggle/working/results") / STEM
def import_run(name):
    complete = [d for d in Path("/kaggle/input").rglob(name)
                if d.is_dir() and (d / "manifest.json").exists()]
    if not complete:
        return None
    with_ckpt = [d for d in complete if (d / "best.pt").exists()]
    src = max(with_ckpt, key=lambda d: (d / "best.pt").stat().st_size) if with_ckpt else complete[0]
    dst = RESULTS / name; dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if f.is_file():
            shutil.copy2(f, dst / f.name)
    return dst

BASE_RUNS = {}
for s in SEEDS:
    d = import_run(f"tcwpn_full_k{K}_seed{s}")
    if d: BASE_RUNS[s] = d
print(f"frozen baseline runs found: {sorted(BASE_RUNS)}")
print("stage A pkl :", PKL_OLD)
print("plans       :", PLAN_DIR)
print("cohort CSV  :", COHORT_CSV)

---

# Part 0 — the two corrections, verified in code

Neither of these needs a GPU or a retrain. Both are checks against the repository source and the
Phase 6/6B logs, and both change what the rest of the notebook does.

In [ ]:
# ---------------------------------------------------------------------------
# 0.2  Correction 1: what does store_fingerprint actually depend on?
# ---------------------------------------------------------------------------
import inspect
from tcwpn.sampler import store_fingerprint, RecordStore, EpisodePlan
print("=" * 78)
print("sampler.store_fingerprint — the function the 'plans must be rebuilt' claim rests on")
print("=" * 78)
print(inspect.getsource(store_fingerprint))

tokc_src = Path("scripts/tokenize_cohort.py").read_text()
m = re.search(r"def chunk_tokenize.*?(?=\ndef )", tokc_src, flags=re.S)
print("=" * 78)
print("tokenize_cohort.chunk_tokenize — what max_chunks actually controls")
print("=" * 78)
print(m.group(0).rstrip() if m else "(not found)")

print("=" * 78)
print("READING")
print("=" * 78)
print("  fingerprint  = f(len(records), note_id of the first 1000 records)")
print("  max_chunks   = how many 512-token windows each record carries")
print("  -> max_chunks changes the CONTENTS of a record, not the record set.")
print("  -> the fingerprint should be UNCHANGED and the frozen plans stay valid.")
print("\n  This is asserted in Part 1, not assumed. If it fails, Part 1 rebuilds the")
print("  plans and re-scores the baseline, which is the expensive path.")

In [ ]:
# ---------------------------------------------------------------------------
# 0.3  Correction 2: the information-matched staircase.
# ---------------------------------------------------------------------------
# Numbers taken from the committed Phase 6 / 6B logs. Nothing is recomputed here.
S = {"S0_pretrained_768": 0.6307, "S1_finetuned_768": 0.7688,
     "S2_projection_256": 0.7611, "S3_tcwpn_full": 0.7379,
     "S2_5shot_centroid_same_episodes": 0.7369}   # Phase 6 uniform-weight counterfactual
N = {"test_notes_all": 3450, "test_notes_queried": 2907, "test_patients": 2278,
     "train_patients_probe": 10290, "support_patients_per_class": K}

print("AS REPORTED (Phase 6B Part C):")
print(f"  S0 pretrained ClinicalBERT 768 : {S['S0_pretrained_768']:.4f}")
print(f"  S1 fine-tuned encoder 768      : {S['S1_finetuned_768']:.4f}  "
      f"({S['S1_finetuned_768']-S['S0_pretrained_768']:+.4f})")
print(f"  S2 projection 256              : {S['S2_projection_256']:.4f}  "
      f"({S['S2_projection_256']-S['S1_finetuned_768']:+.4f})")
print(f"  S3 full TC-WPN                 : {S['S3_tcwpn_full']:.4f}  "
      f"({S['S3_tcwpn_full']-S['S2_projection_256']:+.4f})   <- read as 'ProtoNet wastes the representation'")

print("\nWHY S2 AND S3 ARE NOT COMPARABLE AS MEASURED:")
print(f"  supervision at inference : probe fit on {N['train_patients_probe']:,} labelled patients")
print(f"                             vs {N['support_patients_per_class']} support patients per class")
print(f"  notes per test patient   : probe {N['test_notes_all']/N['test_patients']:.2f} "
      f"(all notes) vs TC-WPN {N['test_notes_queried']/N['test_patients']:.2f} (queried only)"
      f"  = {N['test_notes_all']/N['test_notes_queried']-1:+.1%}")

print("\nINFORMATION-MATCHED COMPARISON (same embeddings, same episodes, same K):")
matched = S["S3_tcwpn_full"] - S["S2_5shot_centroid_same_episodes"]
print(f"  S2 embeddings, {N['train_patients_probe']:,}-patient supervised probe : "
      f"{S['S2_projection_256']:.4f}")
print(f"  S2 embeddings, {K}-shot centroid, same episodes          : "
      f"{S['S2_5shot_centroid_same_episodes']:.4f}")
print(f"  S3 full TC-WPN ({K}-shot + learned weighting)             : "
      f"{S['S3_tcwpn_full']:.4f}")
print(f"\n  matched delta (S3 - matched {K}-shot centroid) = {matched:+.4f}")
print(f"  few-shot protocol cost (supervised probe - {K}-shot centroid) = "
      f"{S['S2_5shot_centroid_same_episodes']-S['S2_projection_256']:+.4f}")

if matched >= 0:
    print("\n  => The prototype stage is NOT below a matched centroid on its own")
    print("     embeddings; it is marginally above it. The 0.0232 'degradation' is")
    print("     the price of few-shot supervision, which is the research design,")
    print("     not a defect to repair.")
    print("     'Investigate the S2 -> S3 gap' is therefore NOT the right Experiment 2.")
json.dump({"as_reported": S, "counts": N, "matched_delta_S3_minus_5shot_centroid": matched},
          open(OUT / "phase7_staircase_correction.json", "w"), indent=2)

---

# Part 1 — rebuild the tokenized data with `max_chunks = 4`

One command, one changed flag. `tokenize_cohort.py` is called with exactly the arguments Stage A
used, except `--max-chunks 4`.

Two things are verified rather than trusted:

1. **Record identity is preserved** — same count, same `note_id`s, same order, same labels. If any
   of that shifts, the comparison is not single-variable and the notebook stops.
2. **The fingerprint matches**, which is what keeps the frozen plans and the frozen baseline valid.

Then the thing the experiment actually depends on: **do the extra chunks contain anything?** If
the median note still fits in one chunk, `max_chunks=4` changes nothing and there is no experiment
to run. The Phase 6 log says 99.9% of notes hit the 512-token ceiling, so this should show real
additional content — but it is measured, because an intervention that turns out to be a no-op
should be discovered here and not after five training runs.

In [ ]:
# ---------------------------------------------------------------------------
# 1.1  Retokenize. ONLY --max-chunks changes.
# ---------------------------------------------------------------------------
MAX_CHUNKS_NEW = 4
PKL_NEW = Path("/kaggle/working/pkl_mc4"); PKL_NEW.mkdir(parents=True, exist_ok=True)

# recover the exact Stage A tokenizer settings from an existing pkl if recorded
import pickle
sample_old = PKL_OLD / f"{STEM}_test.pkl"
print("existing pkl:", sample_old, f"({sample_old.stat().st_size/1e6:.1f} MB)")

!python -m scripts.tokenize_cohort \
    --cohort {COHORT_CSV} \
    --out {PKL_NEW} \
    --max-chunks {MAX_CHUNKS_NEW}

print("\nnew pkl files:")
for p in sorted(PKL_NEW.glob("*.pkl")):
    print(f"   {p.name}  {p.stat().st_size/1e6:.1f} MB")

In [ ]:
# ---------------------------------------------------------------------------
# 1.2  Verify the change is single-variable, and that the plans survive.
# ---------------------------------------------------------------------------
rows, fingerprints_ok = [], True
for split in ("train", "val", "test"):
    old = RecordStore.from_pkl(PKL_OLD / f"{STEM}_{split}.pkl", split_name=split)
    new = RecordStore.from_pkl(PKL_NEW / f"{STEM}_{split}.pkl", split_name=split)

    same_n = len(old.records) == len(new.records)
    same_ids = all(str(a["note_id"]) == str(b["note_id"])
                   for a, b in zip(old.records, new.records))
    same_lab = all(int(a["label"]) == int(b["label"])
                   for a, b in zip(old.records, new.records))
    fp_old, fp_new = store_fingerprint(old), store_fingerprint(new)
    fingerprints_ok &= (fp_old == fp_new)

    ch_old = np.array([len(r["input_ids"]) for r in old.records])
    ch_new = np.array([len(r["input_ids"]) for r in new.records])
    tok_old = np.array([sum(sum(c) for c in r["attention_mask"]) for r in old.records])
    tok_new = np.array([sum(sum(c) for c in r["attention_mask"]) for r in new.records])

    rows.append({"split": split, "n_records": len(new.records),
                 "same_count": same_n, "same_note_ids": same_ids, "same_labels": same_lab,
                 "fingerprint_match": fp_old == fp_new,
                 "chunks_old_mean": ch_old.mean(), "chunks_new_mean": ch_new.mean(),
                 "chunks_new_max": int(ch_new.max()),
                 "tokens_old_mean": tok_old.mean(), "tokens_new_mean": tok_new.mean(),
                 "token_gain_pct": 100 * (tok_new.mean() / max(tok_old.mean(), 1) - 1),
                 "pct_records_with_extra_chunk": 100 * float((ch_new > 1).mean())})

ver = pd.DataFrame(rows)
print(ver.round(3).to_string(index=False))
ver.to_csv(OUT / "phase7_retokenize_verification.csv", index=False)

bad = ver[~(ver.same_count & ver.same_note_ids & ver.same_labels)]
if len(bad):
    raise SystemExit("record identity changed — the intervention is NOT single-variable. STOP.")

print("\n" + "=" * 78)
if fingerprints_ok:
    PLAN_DIR_USE, REBUILD_PLANS = PLAN_DIR, False
    print("FINGERPRINTS MATCH — the frozen episode plans remain valid.")
    print("  The frozen five-seed baseline needs NO re-scoring; the paired comparison")
    print("  in Part 3 runs against the committed numbers directly.")
else:
    PLAN_DIR_USE, REBUILD_PLANS = Path("/kaggle/working/plans_mc4"), True
    print("FINGERPRINTS DIFFER — plans must be rebuilt AND the baseline re-scored")
    print("  on the new plans, or the paired test is invalid. Part 1.3 handles it.")
print("=" * 78)

gain = ver.set_index("split").loc["train", "token_gain_pct"]
print(f"\nadditional text now visible (train): {gain:+.1f}% more tokens per note")
print(f"records gaining a chunk (train)   : "
      f"{ver.set_index('split').loc['train','pct_records_with_extra_chunk']:.1f}%")
if gain < 5:
    print("\n  WARNING: max_chunks=4 barely changes what the model sees. The")
    print("  intervention would be close to a no-op — reconsider before spending")
    print("  five training runs on it.")

In [ ]:
# ---------------------------------------------------------------------------
# 1.3  Only if the fingerprints differed: rebuild plans + re-score the baseline.
# ---------------------------------------------------------------------------
if not REBUILD_PLANS:
    print("skipped — frozen plans are valid.")
else:
    PLAN_DIR_USE.mkdir(parents=True, exist_ok=True)
    !python -m scripts.make_episode_plans \
        --pkl-dir {PKL_NEW} --stem {STEM} --out {PLAN_DIR_USE} --k {K}
    print("\nre-scoring the frozen baseline on the NEW plans (required for a paired test):")
    for s, d in BASE_RUNS.items():
        if not (d / "best.pt").exists():
            print(f"  seed {s}: best.pt missing — cannot re-score"); continue
        !python -m scripts.evaluate --run {d} --split test \
            --pkl-dir {PKL_NEW} --plan-dir {PLAN_DIR_USE} --bootstrap 2000
    print("\nNOTE: re-scoring the OLD checkpoint on 4-chunk pkl feeds it text it was")
    print("never trained on. If plans truly had to be rebuilt, the honest baseline is")
    print("a RETRAINED max_chunks=1 arm on the new plans, not a re-scored old one.")

---

# Part 2 — retrain, five seeds

`scripts.train` unchanged, one config, five seeds, the same K, pointed at the new pkl and the
(same) plans. Nothing else on the command line differs from the Phase 3B baseline runs.

Cost: notes now carry up to 4 chunks, so each forward pass encodes up to 4× the sequences. Expect
substantially longer than the baseline's ~47 min/seed. `RUN_TRAINING` is off by default and the
commands are printed, so a session that runs out of quota can resume seed by seed.

The threshold is selected on validation by `train.py`, exactly as before. **Nothing is selected on
test.**

In [ ]:
# ---------------------------------------------------------------------------
# 2.1  Training commands. Set RUN_TRAINING = True to execute.
# ---------------------------------------------------------------------------
RUN_TRAINING = False
CONFIG = "configs/tcwpn_full.yaml"     # unchanged; the only difference is the pkl
NEW_TAG = f"tcwpn_full_mc{MAX_CHUNKS_NEW}"

cmds = []
for s in SEEDS:
    cmds.append(
        f"python -m scripts.train --config {CONFIG} --k {K} --seed {s} "
        f"--stem {STEM} --pkl-dir {PKL_NEW} --plan-dir {PLAN_DIR_USE} "
        f"--results /kaggle/working/results")
print("EXACT COMMANDS:\n")
print("\n\n".join(cmds))
print(f"\n\nconfig used: {CONFIG}")
print("If that config file does not exist under this name, list configs/ and use the")
print("one the Phase 3B tcwpn_full runs were trained with — the manifest of any")
print("baseline run records it under config.model.preset.")
for d in list(BASE_RUNS.values())[:1]:
    mf = json.loads((d / "manifest.json").read_text())
    print(f"   baseline preset: {mf['config']['model'].get('preset')}")

if RUN_TRAINING:
    for c in cmds:
        print("\n$", c)
        get_ipython().system(c)
else:
    print("\nRUN_TRAINING is False — nothing trained.")

In [ ]:
# ---------------------------------------------------------------------------
# 2.2  Evaluate each new seed on the frozen test plan. Threshold stays locked.
# ---------------------------------------------------------------------------
RUN_EVAL = False
eval_cmds = []
for s in SEEDS:
    run = f"/kaggle/working/results/{STEM}/<config_name>_k{K}_seed{s}"
    eval_cmds.append(f"python -m scripts.evaluate --run {run} --split test "
                     f"--pkl-dir {PKL_NEW} --plan-dir {PLAN_DIR_USE} --bootstrap 2000")
print("\n\n".join(eval_cmds))
if RUN_EVAL:
    for c in eval_cmds:
        get_ipython().system(c)

---

# Part 3 — compare against the frozen baseline

Paired across seeds, because every configuration ran on the same five seeds and the per-seed
difference is the right unit — the same statistic Phase 4 used.

The comparator, from the committed Phase 3B tables:

```text
tcwpn_full  seeds 42-46 : 0.7379, 0.7394, 0.7386, 0.7403, 0.7323   mean 0.7377 ± 0.0031
aux_only    seeds 42-46 : 0.7432, 0.7233, 0.7413, 0.7413, 0.7364   mean 0.7371 ± 0.0081
```

The `aux_only` spread of 0.0199 is the noise floor. Any Δ below it is noise regardless of its
p-value.

The rule is applied mechanically. There is no narrative branch where a Δ of +0.008 becomes "a
promising trend" — it becomes "no effect", and that goes in the paper alongside the Phase 6/6B
diagnosis that explains why.

In [ ]:
# ---------------------------------------------------------------------------
# 3.1  Paired comparison + the pre-registered decision.
# ---------------------------------------------------------------------------
from scipy import stats
BASELINE = {42: 0.7379, 43: 0.7394, 44: 0.7386, 45: 0.7403, 46: 0.7323}
MDE, SEED_SPREAD = 0.020, 0.0199

new_auroc = {}     # {seed: auroc} — fill from each new run's eval_test.json
for s in SEEDS:
    hits = sorted(Path("/kaggle/working/results").rglob(f"*mc*_k{K}_seed{s}/eval_test.json"))
    if hits:
        new_auroc[s] = float(json.loads(hits[0].read_text())["metrics"]["auroc"])

if len(new_auroc) < len(SEEDS):
    print(f"have {len(new_auroc)}/{len(SEEDS)} new seeds: {sorted(new_auroc)}")
    print("Run Part 2 first. The comparison below needs all five.")
    if new_auroc:
        print("\npartial results (NOT a result — do not report or act on these):")
        for s, a in sorted(new_auroc.items()):
            print(f"   seed {s}: {a:.4f}  vs baseline {BASELINE[s]:.4f}  ({a-BASELINE[s]:+.4f})")
else:
    base = np.array([BASELINE[s] for s in SEEDS], float)
    new = np.array([new_auroc[s] for s in SEEDS], float)
    d = new - base
    t_p = float(stats.ttest_rel(new, base).pvalue)
    w_p = float(stats.wilcoxon(new, base).pvalue)
    dz = float(d.mean() / d.std(ddof=1)) if d.std(ddof=1) > 0 else np.nan

    print("per-seed:")
    for s, b, n_ in zip(SEEDS, base, new):
        print(f"   seed {s}: {b:.4f} -> {n_:.4f}   {n_-b:+.4f}")
    print(f"\nmean delta   {d.mean():+.4f}")
    print(f"median delta {np.median(d):+.4f}")
    print(f"seeds better {int((d > 0).sum())}/{len(d)}")
    print(f"paired t p   {t_p:.4f}   wilcoxon p {w_p:.4f}   Cohen's dz {dz:.3f}")
    print(f"\nnoise floor (aux_only seed spread) = {SEED_SPREAD:.4f}")

    success = (d.mean() >= MDE) and (t_p < 0.05) and (int((d > 0).sum()) >= 4)
    print("\n" + "=" * 78)
    print(f"PRE-REGISTERED DECISION: {'SUCCESS' if success else 'NO EFFECT'}")
    print("=" * 78)
    print(f"   mean delta >= +{MDE:.3f}  : {d.mean() >= MDE}")
    print(f"   paired p < 0.05      : {t_p < 0.05}")
    print(f"   >= 4/5 seeds improve : {int((d > 0).sum()) >= 4}")
    if not success:
        print("\n  Report it. A leakage-controlled negative result with a mechanism is")
        print("  a finding; it says the 512-token ceiling is a real pipeline limitation")
        print("  that is NOT the cause of the 0.7379.")
    json.dump({"seeds": SEEDS, "baseline": BASELINE, "new": new_auroc,
               "mean_delta": float(d.mean()), "median_delta": float(np.median(d)),
               "seeds_better": int((d > 0).sum()), "paired_t_p": t_p,
               "wilcoxon_p": w_p, "cohens_dz": dz, "success": bool(success),
               "rule": {"MDE": MDE, "alpha": 0.05, "min_seeds_better": 4}},
              open(OUT / "phase7_result.json", "w"), indent=2)

print("\nAlso run, for the seed-42 pair, the repo's own DeLong test:")
print(f"   python -m scripts.compare_models pair \\")
print(f"       --a <new_run_seed42>/predictions_test.csv \\")
print(f"       --b {BASE_RUNS.get(42, '<frozen_seed42_run>')}/predictions_test.csv")
print("\nAnd check the anxiety-blinded arm: an improvement that vanishes under")
print("blinding is lexical, and the blinding gap is already 0.7379 -> 0.6284.")

---

## What to tell the supervisor

**"Did you investigate?"** — Yes. Phase 6 and 6B cover every branch he named: which notes are
misclassified, whether it is a data problem, whether the embeddings are sound, and whether the
model is at fault. The findings, all leakage-controlled:

- No strong error stratum — k-NN enrichment 1.16× against a 1.5× floor. **Not a data problem** in
  the "collect more of this note type" sense.
- Recency does not explain the errors — every temporal characteristic |δ| < 0.04, Holm p = 1.0.
  Worth saying plainly, because the recency decay was the component's central claim.
- **Bio_ClinicalBERT is not bad.** Pretrained 0.6307 → fine-tuned 0.7688 is +0.1381. Fine-tuning
  adds a great deal; the encoder is doing real work.
- The weighting is active but worth **+0.0009 AUROC** against its own uniform-weight
  counterfactual. It changes 6.24% of wrong decisions without changing discrimination.
- The single data candidate, `anx_coded_this_adm`, is derived from the same ICD diagnoses as the
  label. It cannot be used to rebalance training.

**"Did you fix it?"** — Not yet. Phase 7 is the first controlled attempt, five seeds, one variable.

**Correct the two claims that have crept in:**

1. *"The prototype stage does not exploit the representation"* — as measured, S2 was a probe fit on
   10 290 labelled patients and S3 is a 5-shot method; matched to the same episodes and the same
   embeddings, TC-WPN is +0.0010 above a plain 5-shot centroid. The 0.024 is the cost of few-shot
   learning, not a defect.
2. *"Changing `max_chunks` invalidates the frozen plans"* — `store_fingerprint` hashes record count
   and note_ids only. Part 1 asserts it.

**And on the number itself.** It is **AUROC 0.7379**, not "73.79% accuracy". At the
validation-locked threshold the operating point is sensitivity 0.9217 / specificity 0.2854 — a
screening posture that flags almost every case and accepts many false alarms. For a doctor-facing
supportive tool that may be the right trade, but it has to be stated and justified, not compressed
into one accuracy figure.

**On the 0.80 in the proposal.** He already said missing it is not a big issue. A five-seed,
leakage-certified 0.737 with a mechanism is worth more than an unexplained 0.80. If Phase 7
reaches it honestly, good. If not, report what it gives.

### If Phase 7 shows no effect

Do not change five things. The next question is not "which hyper-parameter" but the one the
matched comparison raises: **is the episodic few-shot protocol the right fit for this task?** The
supervised probe on the same embeddings reaches 0.7611 with full supervision, and there are 10 290
labelled training patients available. A few-shot framing is a design choice, and the honest paper
may be the one that reports what that choice costs.

That is a legitimate contribution: a leakage-controlled benchmark, a negative mechanism result
with a measured explanation, and a quantified cost of the few-shot protocol.